# dots.ocr Test for Dictionary OCR

This notebook tests the dots.ocr model on Assyriological dictionary pages.

**Instructions:**
1. Make sure GPU runtime is enabled: Runtime > Change runtime type > T4 GPU
2. Upload your PNG image(s) when prompted
3. Run all cells

**Note:** For best results, use a GPU with 40GB+ VRAM (A100). T4 requires image resizing which reduces quality.

In [ ]:
# Cell 1 - Install dependencies
!pip install -q transformers accelerate pillow
print("Dependencies installed!")

In [ ]:
# Cell 2 - Check GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM: {vram_gb:.1f} GB")
    if vram_gb < 20:
        print("\n⚠️  Low VRAM - images will be resized (lower quality)")
        print("   For best results, use Colab Pro with A100 GPU")

In [ ]:
# Cell 3 - Upload image(s)
from google.colab import files
from PIL import Image
from IPython.display import display

print("Upload your PNG/JPG image(s):")
uploaded = files.upload()

# Load all uploaded images
images = []
for filename in uploaded.keys():
    img = Image.open(filename).convert("RGB")
    images.append((filename, img))
    print(f"Loaded: {filename} - Size: {img.size}")

# Preview first image
if images:
    print("\nPreview:")
    preview = images[0][1]
    display(preview.resize((400, int(400 * preview.size[1] / preview.size[0]))))

In [ ]:
# Cell 4 - Load dots.ocr model
# Using the repackaged model with working processor (same as HuggingFace Space)
from transformers import AutoModelForCausalLM, AutoProcessor
import time

MODEL_ID = "prithivMLmods/Dots.OCR-Latest-BF16"  # Repackaged version with working processor

print(f"Loading dots.ocr model: {MODEL_ID}")
print("This will take a few minutes to download (~6GB)...")

start = time.time()

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
).eval()

print(f"Model loaded in {time.time() - start:.1f}s")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

In [ ]:
# Cell 5 - OCR Processing Function (with image resizing and repetition fix)
def process_image(image, filename, query="OCR this image."):
    """Process a single image with dots.ocr"""
    print(f"Processing: {filename}...")
    start = time.time()
    
    try:
        torch.cuda.empty_cache()
        
        # Check VRAM and resize if needed
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
        if vram_gb < 20:  # T4 or similar
            max_dim = 1024
        elif vram_gb < 40:  # A10, etc
            max_dim = 1536
        else:  # A100, H100
            max_dim = None  # No resize needed
        
        # Resize if needed
        if max_dim:
            w, h = image.size
            if max(w, h) > max_dim:
                scale = max_dim / max(w, h)
                new_size = (int(w * scale), int(h * scale))
                image = image.resize(new_size, Image.LANCZOS)
                print(f"  Resized: {w}x{h} -> {new_size[0]}x{new_size[1]}")
        
        # Format message (same as HuggingFace Space)
        messages = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": query},
            ]
        }]
        
        # Apply chat template
        prompt = processor.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        # Process inputs
        inputs = processor(
            text=[prompt],
            images=[image],
            return_tensors="pt",
            padding=True
        ).to(model.device)
        
        # Generate (with repetition penalty to prevent loops)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=4096,
                do_sample=True,
                temperature=0.2,
                top_p=0.9,
                top_k=50,
                repetition_penalty=1.2,
            )
        
        # Decode only new tokens
        generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
        result = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Clean up
        result = result.replace("<|im_end|>", "").strip()
        
        processing_time = time.time() - start
        print(f"  Completed in {processing_time:.1f}s")
        
        return {
            "filename": filename,
            "text": result,
            "processing_time_s": processing_time
        }
        
    except Exception as e:
        import traceback
        print(f"  Error: {e}")
        traceback.print_exc()
        return {
            "filename": filename,
            "text": f"ERROR: {str(e)}",
            "processing_time_s": time.time() - start
        }
    finally:
        torch.cuda.empty_cache()

In [ ]:
# Cell 6 - Process all images
results = []
for filename, image in images:
    result = process_image(image, filename)
    results.append(result)

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
total_time = sum(r["processing_time_s"] for r in results)
total_chars = sum(len(r["text"]) for r in results)
print(f"Images processed: {len(results)}")
print(f"Total time: {total_time:.1f}s")
print(f"Total characters: {total_chars}")
if len(results) > 0:
    print(f"Avg time per image: {total_time / len(results):.1f}s")

In [ ]:
# Cell 7 - Display results
for result in results:
    print("\n" + "="*60)
    print(f"{result['filename']} ({result['processing_time_s']:.1f}s)")
    print("="*60)
    print(result["text"][:5000])  # First 5000 chars
    if len(result["text"]) > 5000:
        print(f"\n... [{len(result['text']) - 5000} more characters]")

In [ ]:
# Cell 8 - Save results and download
output_filename = "dots_ocr_results.txt"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write("dots.ocr OCR Results\n")
    f.write(f"Model: {MODEL_ID}\n")
    f.write("="*60 + "\n\n")

    for result in results:
        f.write(f"--- {result['filename']} ({result['processing_time_s']:.1f}s) ---\n\n")
        f.write(result["text"])
        f.write("\n\n")

print(f"Results saved to {output_filename}")
print("Downloading...")
files.download(output_filename)